In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from agent import analyse_news

START = "2026-11-18"
END = "2026-11-25"
print(f"Agent root: {ROOT}")

## Run the representative workflow

The request is for Asterion Technologies (ASTR) over the configured research period.

In [ ]:
report = analyse_news("Asterion Technologies", "ASTR", START, END)
print(report.model_dump_json(indent=2))

## Timeline and key developments

In [ ]:
timeline_rows = []
for date, events in report.timeline.items():
    for event in events:
        timeline_rows.append({
            "date": date,
            "importance": event.importance,
            "category": event.event_category,
            "headline(s)": "; ".join(event.headlines),
            "sources": ", ".join(event.sources),
            "article_ids": ", ".join(event.article_ids),
            "url(s)": ", ".join(url or "missing" for url in event.urls),
        })

pd.DataFrame(timeline_rows)


In [ ]:
pd.DataFrame([development.model_dump() for development in report.key_developments])

## Validation and deployment readiness

The following checks verify notebook outputs, graph entry points, importability and the absence of prohibited advice language.

In [ ]:
from tools.news import advice_policy_violations

output_text = " ".join(
    development.summary + " " + development.why_it_matters
    for development in report.key_developments
)

validation = {
    "ticker_correct": report.ticker == "ASTR",
    "timeline_present": bool(report.timeline),
    "key_developments_have_sources": all(development.source_article_ids for development in report.key_developments),
    "advice_policy_clean": not advice_policy_violations(output_text),
    "category_counts_present": bool(report.category_counts),
}
validation

In [ ]:
config = json.loads((ROOT / "langgraph.json").read_text())
deployment = {
    "langgraph_config_exists": bool(config),
    "news_graph_exported": "lseg_news_agent" in config.get("graphs", {}),
    "dependency_file_exists": (ROOT / "pyproject.toml").exists() or (ROOT / "requirements.txt").exists(),
    "synthetic_feed_exists": (ROOT / "data" / "news.json").exists(),
}
validation.update(deployment)

report_path = ROOT / "notebooks" / "validation_report.json"
report_path.write_text(json.dumps({"checks": validation, "conflicts": [c.model_dump() for c in report.conflicting_signals], "uncertainties": report.uncertainties}, indent=2))
print(json.dumps(validation, indent=2))
print(f"Saved report: {report_path}")

## Summary

- `PASS` means every validation flag in the report is true.
- Conflicting signals and data-quality notes remain visible in the output.
- Results are research analysis, not investment advice.

# LSEG News Agent Demo

Run the synthetic Asterion Technologies news workflow for 18–25 November 2026. This notebook is both a demo and a lightweight validation/deployment readiness check.